## Différentaition automatique.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aderdouri/ql_web_app/blob/master/ql_notebooks/modeling-bonds.ipynb) [![Open In Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/aderdouri/ql_web_app/blob/master/ql_notebooks/modeling-bonds.ipynb)

### Formula:
$$
z = \cos\left(a_0 + \exp(a_1)\right)\left(\sin(a_2) + \cos(a_3)\right) + (a_1)^{\frac{3}{2}} + a_3
$$


/Users/aderdouri/Downloads/EiCNAM/Tutorials/Computational graphs.ipynb

In [ ]:
import tensorflow as tf

# Define the variables
variables = {
    "a0": tf.Variable(1.0, dtype=tf.float32),
    "a1": tf.Variable(2.0, dtype=tf.float32),
    "a2": tf.Variable(3.0, dtype=tf.float32),
    "a3": tf.Variable(4.0, dtype=tf.float32),
}

# Define the function
def func(a0, a1, a2, a3):
    return tf.cos(a0 + tf.exp(a1)) * (tf.sin(a2) + tf.cos(a3)) + tf.pow(a1, 1.5) + a3

# Calculate the gradient
with tf.GradientTape() as tape:
    tape.watch(list(variables.values()))  # Watch the variables
    z = func(*variables.values())

# Compute gradients
grads = tape.gradient(z, list(variables.values()))

# Display gradients as a dictionary
gradient_dict = {name: float(grad.numpy()) for name, grad in zip(variables.keys(), grads)}
print("Computed Gradients:")
for var, grad in gradient_dict.items():
    print(f"{var}: {grad:.6f}")


2024-12-02 17:29:47.455877: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Computed Gradients:
a0: 0.440888
a1: 5.379070
a2: 0.504802
a3: 0.614102


### Black Formula

### Black-Scholes Formula

The Black-Scholes formula for the price of a European call option is given by:

$$
C(S, t) = S\Phi(d_1) - Ke^{-r(T-t)}\Phi(d_2),
$$

where:

$$
d_1 = \frac{\ln(S/K) + \left(r + \frac{\sigma^2}{2}\right)(T-t)}{\sigma\sqrt{T-t}}, \quad
d_2 = d_1 - \sigma\sqrt{T-t}.
$$

#### Parameters:
- C(S, t)\): Call option price at time \(t\),
- \(S\): Current price of the underlying asset,
- \(K\): Strike price of the option,
- \(r\): Risk-free interest rate,
- \(\sigma\): Volatility of the underlying asset,
- \(T\): Time to maturity,
- \($\Phi$($\cdot$)\): Cumulative distribution function of the standard normal distribution.


In [2]:
import tensorflow as tf
import tensorflow_probability as tfp
import math

# Define the Black-Scholes formula as a function
def black_scholes(S, K, r, sigma, T, option_type="call"):
    """
    S: Current stock price
    K: Strike price
    r: Risk-free rate
    sigma: Volatility
    T: Time to maturity
    option_type: "call" or "put"
    """
    d1 = (tf.math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * tf.sqrt(T))
    d2 = d1 - sigma * tf.sqrt(T)

    if option_type == "call":
        price = S * tfp.distributions.Normal(0.0, 1.0).cdf(d1) - K * tf.exp(-r * T) * tfp.distributions.Normal(0.0, 1.0).cdf(d2)
    elif option_type == "put":
        price = K * tf.exp(-r * T) * tfp.distributions.Normal(0.0, 1.0).cdf(-d2) - S * tfp.distributions.Normal(0.0, 1.0).cdf(-d1)
    else:
        raise ValueError("Invalid option_type. Choose 'call' or 'put'.")
    return price

# Parameters
S = tf.Variable(100.0)  # Stock price
K = tf.Variable(100.0)  # Strike price
r = tf.Variable(0.00)   # Risk-free rate
sigma = tf.Variable(0.2)  # Volatility
T = tf.Variable(1.0)    # Time to maturity

# Calculate the derivatives (Greeks) using TensorFlow's GradientTape
with tf.GradientTape(persistent=True) as tape:
    tape.watch([S, K, r, sigma, T])  # Watch all inputs
    price = black_scholes(S, K, r, sigma, T)

# Compute the Greeks
delta = tape.gradient(price, S)   # Sensitivity to stock price
vega = tape.gradient(price, sigma)  # Sensitivity to volatility
theta = tape.gradient(price, T)   # Sensitivity to time to maturity
rho = tape.gradient(price, r)     # Sensitivity to risk-free rate

# Display the results
print(f"Option Price: {price.numpy():.6f}")
print(f"Delta (∂C/∂S): {delta.numpy():.6f}")
print(f"Vega (∂C/∂σ): {vega.numpy():.6f}")
print(f"Theta (∂C/∂T): {theta.numpy():.6f}")
print(f"Rho (∂C/∂r): {rho.numpy():.6f}")


Option Price: 7.965561
Delta (∂C/∂S): 0.539828
Vega (∂C/∂σ): 39.695255
Theta (∂C/∂T): 3.969526
Rho (∂C/∂r): 46.017220


In [5]:
import tensorflow as tf
import numpy as np

# SABR model parameters
F = tf.Variable(100.0, dtype=tf.float32)  # Forward price
K = tf.Variable(105.0, dtype=tf.float32)  # Strike price
alpha = tf.Variable(0.2, dtype=tf.float32)  # Initial volatility (α)
beta = tf.Variable(0.5, dtype=tf.float32)  # Elasticity parameter (β)
nu = tf.Variable(0.3, dtype=tf.float32)  # Volatility of volatility (ν)
rho = tf.Variable(-0.1, dtype=tf.float32)  # Correlation (ρ)
T = tf.Variable(1.0, dtype=tf.float32)  # Time to maturity

# SABR implied volatility approximation
def sabr_implied_vol(F, K, alpha, beta, nu, rho, T):
    z = (nu / alpha) * ((F * K) ** ((1 - beta) / 2)) * tf.math.log(F / K)
    x_z = tf.math.log((tf.sqrt(1 - 2 * rho * z + z**2) + z - rho) / (1 - rho))
    factor1 = alpha / ((F * K) ** ((1 - beta) / 2))
    factor2 = (1 + ((1 - beta) ** 2 / 24) * (tf.math.log(F / K)**2))
    return factor1 * (z / x_z) * factor2

# Compute the implied volatility
with tf.GradientTape(persistent=True) as tape:
    implied_vol = sabr_implied_vol(F, K, alpha, beta, nu, rho, T)

# First-order sensitivities
delta = tape.gradient(implied_vol, F)  # ∂σ/∂F
vega = tape.gradient(implied_vol, nu)  # ∂σ/∂ν
rho_sensitivity = tape.gradient(implied_vol, rho)  # ∂σ/∂ρ
beta_sensitivity = tape.gradient(implied_vol, beta)  # ∂σ/∂β
alpha_sensitivity = tape.gradient(implied_vol, alpha)  # ∂σ/∂α
theta = tape.gradient(implied_vol, T)  # ∂σ/∂T

# Second-order sensitivities
gamma = tape.gradient(delta, F)  # ∂²σ/∂F²
volga = tape.gradient(vega, nu)  # ∂²σ/∂ν²

# Release tape resources
del tape

# Display all results
print(f"Implied Volatility: {implied_vol.numpy():.6f}")
print(f"Delta (∂σ/∂F): {delta.numpy():.6f}" if delta is not None else "Delta: None")
print(f"Vega (∂σ/∂ν): {vega.numpy():.6f}" if vega is not None else "Vega: None")
print(f"Rho Sensitivity (∂σ/∂ρ): {rho_sensitivity.numpy():.6f}" if rho_sensitivity is not None else "Rho Sensitivity: None")
print(f"Beta Sensitivity (∂σ/∂β): {beta_sensitivity.numpy():.6f}" if beta_sensitivity is not None else "Beta Sensitivity: None")
print(f"Alpha Sensitivity (∂σ/∂α): {alpha_sensitivity.numpy():.6f}" if alpha_sensitivity is not None else "Alpha Sensitivity: None")
print(f"Theta (∂σ/∂T): {theta.numpy():.6f}" if theta is not None else "Theta: None")
print(f"Gamma (∂²σ/∂F²): {gamma.numpy():.6f}" if gamma is not None else "Gamma: None")
print(f"Volga (∂²σ/∂ν²): {volga.numpy():.6f}" if volga is not None else "Volga: None")


Implied Volatility: 0.020716
Delta (∂σ/∂F): -0.000530
Vega (∂σ/∂ν): 0.007878
Rho Sensitivity (∂σ/∂ρ): 0.006446
Beta Sensitivity (∂σ/∂β): 0.084963
Alpha Sensitivity (∂σ/∂α): 0.091764
Theta: None
Gamma: None
Volga: None


In [6]:
import tensorflow as tf

# Constants for the Java-like implementation
Z_RANGE = 1e-5  # Threshold for small z approximation

# SABR normal approximation function
def sabr_normal_vol(F, K, alpha, beta, rho, nu, expiry):
    """
    TensorFlow implementation of SABR Normal Approximation based on the provided Java code.
    Parameters:
    - F: Forward price
    - K: Strike price
    - alpha: Initial volatility (α)
    - beta: Elasticity parameter (must be 0 for the normal model)
    - rho: Correlation between forward and volatility
    - nu: Volatility of volatility
    - expiry: Time to maturity
    """
    if beta != 0:
        raise ValueError("Beta must be 0 for the normal approximation.")

    beta1 = 1.0 - beta
    fKbeta = (F * K) ** (0.5 * beta1)
    logfK = tf.math.log(F / K)
    z = (nu / alpha) * fKbeta * logfK

    # Approximation for small z
    zxz = tf.where(
        tf.abs(z) < Z_RANGE,
        1.0 - 0.5 * z * rho,  # Small z approximation
        z / tf.math.log((tf.sqrt(1.0 - 2.0 * rho * z + z**2) + z - rho) / (1.0 - rho))
    )

    beta24 = beta1**2 / 24.0
    beta1920 = beta1**4 / 1920.0
    logfK2 = logfK**2

    # Adjustment factors
    factor11 = beta24 * logfK2
    factor12 = beta1920 * logfK2**2
    num1 = 1.0 + factor11 + factor12
    factor1 = alpha / (fKbeta * num1)

    factor31 = beta24 * alpha**2 / fKbeta**2
    factor32 = 0.25 * rho * beta * nu * alpha / fKbeta
    factor33 = (2.0 - 3.0 * rho**2) / 24.0 * nu**2
    factor3 = 1.0 + (factor31 + factor32 + factor33) * expiry

    # Final volatility
    return factor1 * zxz * factor3

# Model parameters
F = tf.Variable(100.0, dtype=tf.float32)  # Forward price
K = tf.Variable(105.0, dtype=tf.float32)  # Strike price
alpha = tf.Variable(0.2, dtype=tf.float32)  # Initial volatility (α)
beta = tf.Variable(0.5, dtype=tf.float32)  # Elasticity parameter (must be 0 for normal approximation)
rho = tf.Variable(-0.1, dtype=tf.float32)  # Correlation (ρ)
nu = tf.Variable(0.3, dtype=tf.float32)  # Volatility of volatility (ν)
expiry = tf.Variable(1.0, dtype=tf.float32)  # Time to maturity

# Compute implied volatility and sensitivities
with tf.GradientTape(persistent=True) as tape:
    implied_vol = sabr_normal_vol(F, K, alpha, beta, rho, nu, expiry)

# Calculate sensitivities (Greeks)
delta = tape.gradient(implied_vol, F)  # ∂σ/∂F
vega = tape.gradient(implied_vol, nu)  # ∂σ/∂ν
rho_sensitivity = tape.gradient(implied_vol, rho)  # ∂σ/∂ρ
alpha_sensitivity = tape.gradient(implied_vol, alpha)  # ∂σ/∂α
theta = tape.gradient(implied_vol, expiry)  # ∂σ/∂T

# Release the persistent tape
del tape

# Display the results
print(f"Implied Volatility: {implied_vol.numpy():.6f}")
print(f"Delta (∂σ/∂F): {delta.numpy():.6f}" if delta is not None else "Delta: None")
print(f"Vega (∂σ/∂ν): {vega.numpy():.6f}" if vega is not None else "Vega: None")
print(f"Rho Sensitivity (∂σ/∂ρ): {rho_sensitivity.numpy():.6f}" if rho_sensitivity is not None else "Rho Sensitivity: None")
print(f"Alpha Sensitivity (∂σ/∂α): {alpha_sensitivity.numpy():.6f}" if alpha_sensitivity is not None else "Alpha Sensitivity: None")
print(f"Theta (∂σ/∂T): {theta.numpy():.6f}" if theta is not None else "Theta: None")


Implied Volatility: 0.005257
Delta (∂σ/∂F): -0.000701
Vega (∂σ/∂ν): 0.011505
Rho Sensitivity (∂σ/∂ρ): 0.001842
Alpha Sensitivity (∂σ/∂α): 0.009415
Theta (∂σ/∂T): 0.000039


In [ ]:
Calculated volatility: 0.0208666
Calculated volatility with AAD: 0.0208666
Derivatives: -0.00053389
0.0924239
0.0855752
0.00655436
0.00894985
0.000420444
0.000151581